# 4 — RL + Attention NLP (DQN + Actor-Critic) with Banded Bagging

Pool of RL agents whose *policy/value nets contain attention over the NLP state*
(e.g. state = post text / coaching history / user goal). Mix of DQN (value-based) and
Actor-Critic/A2C (policy-based) members. Each trains independently; when episodic return /
success-rate crosses a band (`37..81..90%` of oracle/normalised score) it joins the band pool,
waits the grace period, then contributes to the **bagged behaviour policy** (majority vote /
averaged advantages). Same bias logic as notebooks 1–2: don't let one lucky agent dominate early.

**BuddyUp mapping:** coaching-cue selection, workout-progression actions, reply moderation
actions. v1 prod equivalent is already LinUCB in `feed_ranking.ipynb` — promote to this only
with real engagement trajectories.

In [1]:
import importlib.util, os, pathlib, sys
p = pathlib.Path(os.getcwd()).resolve()
ai = None
while p != p.parent:
    for cand in (p / 'backend' / 'ai_service', p / 'ai_service', p):
        if (cand / 'training').is_dir() and (cand / 'notebooks').is_dir():
            ai = cand; break
    if ai is not None: break
    p = p.parent
if ai is None: raise RuntimeError('ai_service not found')
sys.path.insert(0, str(ai / 'training')); sys.path.insert(0, str(ai))
os.chdir(ai / 'notebooks')
# ── load the repo .env (BUDDY_SCALE, KAGGLE_API_TOKEN, …) BEFORE bootstrap ──
# bootstrap reads BUDDY_SCALE at import time, so this must run first. Uses
# python-dotenv when available, else a tiny built-in parser (Kaggle-safe).
def _find_dotenv(start):
    p = pathlib.Path(start).resolve()
    while p != p.parent:
        f = p / '.env'
        if f.is_file():
            return f
        p = p.parent
    return None

_env_file = _find_dotenv(ai)
try:
    from dotenv import load_dotenv
    load_dotenv(_env_file)
except ImportError:
    if _env_file:
        for _line in _env_file.read_text().splitlines():
            _line = _line.strip()
            if not _line or _line.startswith('#') or '=' not in _line:
                continue
            _k, _, _v = _line.partition('=')
            os.environ.setdefault(_k.strip(), _v.strip().strip('"').strip("'"))
print('[env]', _env_file or 'no .env found', '| BUDDY_SCALE =', os.environ.get('BUDDY_SCALE'),
      '| KAGGLE_API_TOKEN =', 'set' if os.environ.get('KAGGLE_API_TOKEN') else 'missing')
_missing = [m for m in ['torch'] if importlib.util.find_spec(m) is None]
if _missing:
    get_ipython().run_line_magic('pip', 'install -q ' + ' '.join(_missing))
# bootstrap.py reads BUDDY_SCALE at import time; if an earlier run in this
# same kernel cached the module (e.g. before the .env was loaded), drop the
# stale copy so the current environment is honoured.
for _stale in ('training.bootstrap', 'bootstrap'):
    _m = sys.modules.get(_stale)
    if _m is not None and getattr(_m, 'BUDDY_SCALE', None) != os.environ.get('BUDDY_SCALE'):
        sys.modules.pop(_stale, None)
        print(f'[bootstrap] re-importing {_stale} (stale scale cache cleared)')
try:
    from training.bootstrap import *
    CFG = init(scale=os.environ.get('BUDDY_SCALE') or None)
except Exception as e:
    print('[bootstrap] unavailable:', e); CFG = {}
except Exception as e:
    print('[bootstrap] unavailable:', e); CFG = {}
SCALE = CFG.get('scale', os.environ.get('BUDDY_SCALE', 'demo'))
print('scale:', SCALE)

[env] /home/peter/Desktop/Buddy-Up/backend/.env | BUDDY_SCALE = smoke | KAGGLE_API_TOKEN = set


2026-09-16 16:45:43.936041: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-16 16:45:44.155977: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


2026-09-16 16:45:47.874726: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


scale: smoke


In [2]:
import torch, torch.nn as nn, numpy as np, random
torch.manual_seed(2); np.random.seed(2); random.seed(2)
BANDS = [0.37, 0.47, 0.57, 0.67, 0.71, 0.73, 0.77, 0.79, 0.81, 0.85, 0.90]
GRACE_EP = {'smoke': 2, 'demo': 5, 'full': 20}[SCALE]
N_AGENTS = {'smoke': 4, 'demo': 6, 'full': 12}[SCALE]  # half DQN, half A2C

class AttnState(nn.Module):
    """Attention over a sequence of text-token features -> fixed state vector."""
    def __init__(self, tok=32, d=64):
        super().__init__()
        self.emb = nn.Embedding(500, d)
        self.attn = nn.MultiheadAttention(d, 4, batch_first=True)
        self.proj = nn.Linear(d, tok)
    def forward(self, toks):  # (B, T)
        h = self.emb(toks)
        a, _ = self.attn(h, h, h)
        return a.mean(1)

class DQNAgent(nn.Module):
    def __init__(self, na=4):
        super().__init__(); self.kind = 'dqn'
        self.enc = AttnState(); self.q = nn.Linear(64, na)
    def act(self, s, eps):
        import random as r
        if r.random() < eps: return r.randrange(self.q.out_features)
        with torch.no_grad(): return int(self.q(self.enc(s)).argmax(1))

class A2CAgent(nn.Module):
    def __init__(self, na=4):
        super().__init__(); self.kind = 'a2c'
        self.enc = AttnState(); self.pi = nn.Linear(64, na); self.v = nn.Linear(64, 1)
    def act(self, s):
        with torch.no_grad(): return int(torch.softmax(self.pi(self.enc(s)), -1).multinomial(1))

In [3]:
# Toy env: state = token seq hinting the correct action; reward 1 if action == hidden label.
# Stand-in for: state=user history text, action=which workout/cue/reply to push.
class TextBandit:
    def __init__(self, na=4, T=16): self.na, self.T = na, T
    def reset(self):
        if _RB is not None:  # REAL states: sampled IRL titles, rule labels
            j = int(np.random.randint(len(_RB['X'])))
            self.label = int(_RB['y'][j]); self.s = _RB['X'][j].unsqueeze(0)
            return self.s
        self.label = np.random.randint(self.na)
        toks = np.random.randint(0, 500, self.T); toks[0] = self.label * 100  # weak signal
        self.s = torch.tensor(toks).unsqueeze(0)
        return self.s
    def step(self, a): return self.s, float(a == self.label), True, {}

env = TextBandit()
# --- real batch (BUDDY_BATCH): states become real IRL titles ---
from batch_data import has_batch, batch_meta, load_tensors
_RB = None
if has_batch():
    _m = batch_meta(); _RB = load_tensors('X', 'y')
    print('REAL states', tuple(_RB['X'].shape), '|', _m['source'])
agents = [(DQNAgent() if i % 2 == 0 else A2CAgent()) for i in range(N_AGENTS)]
opts = [torch.optim.Adam(a.parameters(), lr=2e-3) for a in agents]
EPISODES = {'smoke': 60, 'demo': 300, 'full': 2000}[SCALE]
scores = {i: [] for i in range(N_AGENTS)}; joined = {}
import torch.nn.functional as F
for ep in range(EPISODES):
    eps = max(0.05, 1.0 - ep / (EPISODES * 0.7))
    for i, ag in enumerate(agents):
        s = env.reset()
        if ag.kind == 'dqn':
            a = ag.act(s, eps); _, r, _, _ = env.step(a)
            q = ag.q(ag.enc(s))[0, a]
            loss = (q - torch.tensor(r)) ** 2  # 1-step target (toy)
            opts[i].zero_grad(); loss.backward(); opts[i].step()
        else:
            logits = ag.pi(ag.enc(s)); v = ag.v(ag.enc(s))
            dist = torch.distributions.Categorical(logits=logits)
            a = dist.sample(); _, r, _, _ = env.step(int(a))
            adv = torch.tensor(float(r)) - v.detach()
            loss = -(dist.log_prob(a) * adv) + F.mse_loss(v, torch.tensor([[float(r)]]))
            opts[i].zero_grad(); loss.backward(); opts[i].step()
        scores[i].append(r)
    if (ep + 1) % 20 == 0:  # band gate on rolling success rate (normalised score)
        for i in range(N_AGENTS):
            acc = float(np.mean(scores[i][-100:]))
            b = max([t for t in BANDS if acc >= t], default=None)
            if b and i not in joined:
                joined[i] = (b, ep); print(f'ep{ep}: agent {i}({agents[i].kind}) win-rate={acc:.2f} joins {int(b*100)}%')
print('joined:', joined)

REAL states (500000, 16) | meirl titles, rule labels (?/!/long/other)


ep19: agent 2(dqn) win-rate=0.60 joins 56%
ep19: agent 3(a2c) win-rate=0.50 joins 47%


ep39: agent 0(dqn) win-rate=0.40 joins 37%
ep39: agent 1(a2c) win-rate=0.47 joins 47%


joined: {2: (0.57, 19), 3: (0.47, 19), 0: (0.37, 39), 1: (0.47, 39)}


In [4]:
# Bagged behaviour policy: majority vote among grace-eligible members vs best single
import numpy as np
elig = [i for i, (_, e0) in joined.items() if EPISODES - e0 >= GRACE_EP]
if not elig: elig = list(range(len(agents)))  # fall back pre-grace
wins = []
for _ in range(200):
    s = env.reset()
    votes = []
    for i in elig:
        ag = agents[i]; ag.eval()
        with torch.no_grad():
            if ag.kind == 'dqn': votes.append(int(ag.q(ag.enc(s)).argmax(1)))
            else: votes.append(int(ag.pi(ag.enc(s)).argmax(1)))
    a = max(set(votes), key=votes.count); _, r, _, _ = env.step(a)
    wins.append(r)
print(f'eligible={elig} bagged win-rate={np.mean(wins):.3f}')
# Export one actor head (serving: state tokens -> action logits)
a2c = next(a for a in agents if a.kind == 'a2c'); a2c.eval()
class PolicyOnly(nn.Module):
    def __init__(self, a): super().__init__(); self.e = a.enc; self.p = a.pi
    def forward(self, t): return self.p(self.e(t))
torch.onnx.export(PolicyOnly(a2c), torch.randint(0, 500, (1, 16)), '../models/rl_nlp_policy.onnx',
    input_names=['tokens'], output_names=['action_logits'], dynamic_axes={'tokens': {0: 'batch'}})
print('exported ../models/rl_nlp_policy.onnx')

eligible=[2, 3, 0, 1] bagged win-rate=1.000


/tmp/ipykernel_2891509/2240467770.py:22: UserWarning: Exporting a model while it is in training mode. Please ensure that this is intended, as it may lead to different behavior during inference. Calling model.eval() before export is recommended.
  torch.onnx.export(PolicyOnly(a2c), torch.randint(0, 500, (1, 16)), '../models/rl_nlp_policy.onnx',
/tmp/ipykernel_2891509/2240467770.py:22: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(PolicyOnly(a2c), torch.randint(0, 500, (1, 16)), '../models/rl_nlp_policy.onnx',


[torch.onnx] Obtain model graph for `PolicyOnly([...]` with `torch.export.export(..., strict=False)`...


[torch.onnx] Obtain model graph for `PolicyOnly([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...


[torch.onnx] Optimize the ONNX graph... ✅
exported ../models/rl_nlp_policy.onnx


## Data, scrapers vs platforms (notebook 4)

| Need | Recommendation |
|---|---|
| Env data (preferred) | Start with Gymnasium (`CartPole`, `FrozenLake`) + RecSim / RecoGym for rec-style RL, then BuddyUp `engagement.csv` trajectories (state=history, action=shown post, reward=click/dwell). No scraper needed — logs beat crawls. |
| NLP state text | Same corpora as notebook 1; HF `transformers` tokenizers. |
| Scrapers/bots | Only to *backfill item metadata* (product/post text). Never to fake reward labels. |
| Platforms | CleanRL / Stable-Baselines3 (reference algos), Ray RLlib (multi-agent scale-out — one worker per band member), W&B (return curves per agent), SageMaker RL / Vertex (managed training). DQN+Actor-Critic are in SB3 already; this notebook adds the attention-state + banded bagging on top. |